# Module 8 · LLM Observability & Cost Management

**Phase:** Production Engineering  
**Toolchain:** Langfuse · LangSmith · Token Budgeting · Cost Dashboards  
**Objective:** Learn how to instrument your AI systems with telemetry to monitor traces, debug complex reasoning pipelines, and surgically control API costs.

---

## 🧠 Why LLM Observability Is Fundamentally Different

Traditional software monitoring tracks latency, error rates, and CPU throughput.  
LLM monitoring tracks **tokens, subjective quality, hallucinations, tool-use execution graphs, and prompt versions.**

| Traditional Monitoring | LLM Observability |
|----------------------|-----------------|
| Response time | Response time + Time-to-First-Token (TTFT) |
| HTTP Error Rate | HTTP Errors + Hallucination Rate (Soft errors) |
| Server cost | LLM API cost per specific trace |
| Route coverage | Prompt version tracking |

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** You cannot optimize what you cannot trace. Seniors instrument and observe every LLM interaction, tracing the exact sequence, token count, latency, and cost of all sub-components (Retrievers, Tools, LLMs) to identify bottlenecks and stop runaway spending loops.

**Mental Model:** Advanced LLM Observability (like Langfuse) acts as a highly detailed itemized receipt. It shows you exactly how much the 'Vector Search' step and the 'Context Re-ranking' step contributed to the final $0.05 bill and 2.5s latency of the response.

**Common Junior Pitfall:** Deploying agents or RAG pipelines without token budgets or node tracking. A runaway recursive agent loop can rack up thousands of dollars in API fees overnight, and without tracing, you won't know why it failed or why it cost so much.

---


## 📑 Table of Contents

* [1 · Production Request Tracing (Langfuse Pattern)](#1--production-request-tracing-langfuse-pattern)
  * [🤔 What is a Trace?](#what-is-a-trace)
* [2 · Token Budgeting: Preventing Bankruptcy](#2--token-budgeting-preventing-bankruptcy)
  * [🤔 The Hierarchy of Budgeting](#the-hierarchy-of-budgeting)
* [3 · Prompt Version Management](#3--prompt-version-management)
* [✅ Knowledge Check](#-knowledge-check)
* [🔨 Practical Practice](#-practical-practice)
* [🎯 Summary & Key Takeaways](#-summary--key-takeaways)


---
## 1 · Production Request Tracing (Langfuse Pattern)

### 🤔 What is a Trace?

A **trace** records everything that happens chronologically during a single LLM pipeline invocation. Modern tools like **Langfuse** or **LangSmith** use OpenTelemetry standards to map this out.

```
Trace: "user asked about Q4 revenue"
  ├── Span: Input guardrails     (2ms, passed)
  ├── Span: RAG retrieval        (150ms, 5 chunks found)
  ├── Span: Prompt assembly      (1ms, 1,200 input tokens)
  ├── Span: LLM call             (2,100ms, 350 output tokens, $0.012)
  └── Span: Output guardrails    (5ms, passed)
  
  Total: 2,261ms | Cost: $0.012 | Quality: 0.92
```

Below is a demonstration of how a trace object is structured and reported to an observability backend.


In [ ]:
# Cell 1 -- LLM Tracing Simulation (Langfuse Style)
import time
import uuid

class TraceSpan:
    def __init__(self, name, parent=None):
        self.id = str(uuid.uuid4())[:8]
        self.name = name
        self.parent = parent
        self.start_time = time.time()
        self.end_time = None
        self.metadata = {}
        self.metrics = {'tokens_in': 0, 'tokens_out': 0, 'cost': 0.0}

    def end(self):
        self.end_time = time.time()
        return (self.end_time - self.start_time) * 1000  # ms

    def add_metric(self, tokens_in, tokens_out, cost):
        self.metrics['tokens_in'] = tokens_in
        self.metrics['tokens_out'] = tokens_out
        self.metrics['cost'] = cost

class LangfuseSimulator:
    def __init__(self):
        self.traces = []

    def start_trace(self, name):
        trace = TraceSpan(name)
        self.traces.append(trace)
        return trace

# SIMULATION
obs = LangfuseSimulator()
trace = obs.start_trace("RAG_Query_Financials")

print(f"\ud83d\udd0e Trace started: {trace.name} (ID: {trace.id})")

# Span 1
span_rag = TraceSpan("VectorDB_Retrieve", parent=trace.id)
time.sleep(0.15) # mock DB delay
duration1 = span_rag.end()
print(f"  ├── [Span] {span_rag.name}: {duration1:.1f}ms (Retrieved 5 chunks)")

# Span 2
span_llm = TraceSpan("OpenAI_GPT4o_Call", parent=trace.id)
time.sleep(0.4) # mock generation delay
span_llm.add_metric(tokens_in=1200, tokens_out=350, cost=0.012)
duration2 = span_llm.end()
print(f"  ├── [Span] {span_llm.name}: {duration2:.1f}ms | Cost: ${span_llm.metrics['cost']}")

total_duration = trace.end()
print(f"  └── Trace Complete | Total Latency: {total_duration:.1f}ms")
print("\nThis structured JSON is sent asynchronously to the Langfuse backend.")


---
## 2 · Token Budgeting: Preventing Bankruptcy

### 🤔 The Hierarchy of Budgeting

Without budgets, one heavy user or a recursive error loop can spend your entire monthly API budget in a day. You must implement budgets at the gateway level.

| Budget Level | What It Controls | Example |
|-------------|-----------------|--------|
| **Per-request** | Max tokens per single request | `max_tokens=4000` |
| **Per-user/day** | Daily spending limit per user | $5.00/day per user |
| **Per-team/month** | Monthly budget per team | $500/month for engineering |

Below is a demonstration of an API gateway interceptor that enforces hard spending limits *before* the request reaches OpenAI.


In [ ]:
# Cell 2 -- Token gateway budgeting loop
class BudgetGateway:
    def __init__(self):
        self.usage = {}
        self.limits = {'user_alice': 0.05, 'user_bob': 10.00}

    def mock_llm_call(self, user_id, estimated_cost):
        current_usage = self.usage.get(user_id, 0.0)
        limit = self.limits.get(user_id, 0.0)

        if current_usage + estimated_cost > limit:
            print(f"\u274c BLOCKED [{user_id}]: Daily budget exceeded. Limit: ${limit}. Current: ${current_usage:.3f}")
            return False
        
        # Proceed with call
        self.usage[user_id] = current_usage + estimated_cost
        print(f"\u2705 ALLOWED [{user_id}]: Consumed ${estimated_cost:.3f}. New Total: ${self.usage[user_id]:.3f}")
        return True

gateway = BudgetGateway()
gateway.mock_llm_call("user_alice", 0.02)
gateway.mock_llm_call("user_alice", 0.04)
gateway.mock_llm_call("user_bob", 0.04)


---
## 3 · Prompt Version Management

In standard software development, code is version-controlled via Git. In LLM applications, the Prompt is also code. 

Observability tools allow you to tie specific metric outcomes to specific prompt permutations:
```
Prompt v1 (baseline):  Quality = 0.82, Cost = $0.015/req
Prompt v2 (improved):  Quality = 0.91, Cost = $0.018/req  ← Deploy this
Prompt v3 (bad edit):  Quality = 0.65, Cost = $0.020/req  ← Auto-rollback!
```


---
## ✅ Knowledge Check

### Q1: Latency Metrics
Why do we track 'Time-to-First-Token' (TTFT) separately from total response time in LLM applications?
<details><summary>Answer</summary>
Because LLMs generate tokens autoregressively, users perceive performance primarily via how fast the text *starts* streaming on their screen. A 5-second total response time is entirely acceptable if the TTFT was 300ms (the user reads as it types). If TTFT is 4 seconds, the app feels broken, even though the total latency is the same. 
</details>

### Q2: Prompt Regression
How does production LLM observability differ from just using a notebook to test your prompts?
<details><summary>Answer</summary>
A notebook shows you performance on a tiny, curated, static dataset. Production observability (like Langfuse traces) aggregates continuous live usage, letting you see if a new prompt version causes quality regression or a massive spike in token costs over real-world edge cases.
</details>

---
## 🔨 Practical Practice

### Exercise 1: Cost Calculator
Assuming `gpt-4o` costs $5.00 per 1M input tokens and $15.00 per 1M output tokens, write a Python function that takes a traced payload (`tokens_in=3420, tokens_out=812`) and returns the exact decimal cost of the execution step.

### Exercise 2: Implementing Sub-Spans
Enhance the `LangfuseSimulator` class above to support infinite tree nesting (sub-spans within sub-spans), so you can accurately trace an Agent calling a Tool, which subsequently invokes a Database Query.


---
## 🎯 Summary & Key Takeaways

| Practice | What It Solves | Tool |
|----------|---------------|------|
| Request tracing | "Why was this specific 5-tool sequence slow/wrong?" | Langfuse, LangSmith |
| Token budgeting | "Who spent all our API budget while I slept?" | Custom Middleware/Gateway |
| Prompt versioning | "Which prompt version caused hallucination spikes?" | Langfuse, W&B |
| Cost edge-tracking | "How much are we spending per enterprise tenant?" | Helicone, custom |

### 🧠 The 5 Observability Rules
1. **Trace every request** — you can't debug a black box API.
2. **Set budgets BEFORE launch** — not after the $10,000 billing surprise.
3. **Version all prompts** — so you can scientifically compare and instantly rollback.
4. **Alert on TTFT** — user experience dies if Time-to-First-Token exceeds 1.5 seconds.
5. **Monitor by Tenant/User** — identify your most expensive users natively.

**Next →** `39_llm_evaluation.ipynb` — Module 7 — Advanced LLM Evaluation Frameworks & Traceability
